# Homework 1 — Scientific Paper Processing with spaCy

This notebook implements a complete NLP pipeline that transforms scientific PDF documents into structured linguistic data. The pipeline processes all nine supplied papers with the same code:

**PDF → Text → Sections → Paragraphs → Sentences → Tokens → Linguistic Annotations**

Each task builds on the previous one, and the final output is a nested JSON file per paper suitable for Homework 2.

In [1]:
import sys
from pathlib import Path

# Resolve project root whether running from notebooks/ or project root
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source path added: {PROJECT_ROOT / 'src'}")

Project root: /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2
Source path added: /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/src


---
## Task 1 — PDF Text Extraction

We use **PyMuPDF** to extract text page by page. The `sort_blocks=True` parameter sorts text blocks by their position on the page, which helps with reading order on single-column pages but can still mix columns on two-column layouts.

The extraction preserves:
- **Page boundaries** — each page becomes a separate record
- **Line structure** — newlines from the PDF are kept
- **Paragraphs** — blank lines between text blocks are preserved

In [2]:
from scipaper.extraction import extract_pdf, document_statistics, save_extracted_text

PAPER_DIR = PROJECT_ROOT / 'data' / 'papers'
PDF_FILES = sorted(PAPER_DIR.glob('*.pdf'))
SORT_BLOCKS = True
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'extracted_text'

missing = [str(p) for p in PDF_FILES if not p.is_file()]
if missing:
    print('WARNING — PDF paths not found:', missing)
else:
    print(f'{len(PDF_FILES)} PDFs ready from {PAPER_DIR}')
    for p in PDF_FILES:
        print(f'  • {p.name}')

9 PDFs ready from /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/data/papers
  • 1406.1078v3.pdf
  • 1409.0473v7.pdf
  • 1412.3555v1.pdf
  • 1601.06733v7.pdf
  • 1607.06450v1.pdf
  • 1610.02357v3.pdf
  • 1703.03906v2.pdf
  • AttentionYouNeed.pdf
  • N16-1024.pdf


In [3]:
# Extract text from all PDFs and save page-marked text files
results = []
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    text_path = OUTPUT_DIR / f'{pdf_path.stem}.txt'
    save_extracted_text(pages, text_path)
    stats = document_statistics(pages)
    results.append({'paper': pdf_path.stem, **stats, 'text_file': str(text_path)})
    print(f"{pdf_path.stem}: {stats['pages']} pages, {stats['characters']:,} chars, {stats['words']:,} words")

print(f"\nTotal: {sum(r['pages'] for r in results)} pages, "
      f"{sum(r['characters'] for r in results):,} characters, "
      f"{sum(r['words'] for r in results):,} words")

1406.1078v3: 15 pages, 66,324 chars, 7,804 words


1409.0473v7: 15 pages, 60,133 chars, 8,311 words


1412.3555v1: 9 pages, 28,097 chars, 3,967 words


1601.06733v7: 11 pages, 58,932 chars, 7,024 words


1607.06450v1: 14 pages, 62,658 chars, 7,698 words


1610.02357v3: 8 pages, 38,972 chars, 4,474 words


1703.03906v2: 9 pages, 47,065 chars, 5,424 words


AttentionYouNeed: 15 pages, 44,623 chars, 6,240 words


N16-1024: 11 pages, 58,929 chars, 6,784 words

Total: 107 pages, 465,733 characters, 57,726 words


In [4]:
# Inspect short excerpts from three pages of the main paper
pages = extract_pdf(PAPER_DIR / 'AttentionYouNeed.pdf', sort_blocks=SORT_BLOCKS)
for page_idx in [0, 2, 5]:  # pages 1, 3, 6
    print(f'\n{"="*60}')
    print(f'PAGE {pages[page_idx]["page"]}')
    print(f'{"="*60}')
    print(pages[page_idx]["text"][:800])


PAGE 1
Provided proper attribution is provided, Google hereby grants permission to
                     reproduce the tables and figures in this paper solely for use in journalistic or
                                                     scholarly works.


                          Attention Is All You Need





                         Ashish Vaswani∗      Noam Shazeer∗       Niki Parmar∗     Jakob Uszkoreit∗2023                     Google Brain          Google Brain       Google Research     Google Research
Aug               avaswani@google.comLlion Jones∗      noam@google.comAidan N. Gomez∗†nikip@google.comŁukaszusz@google.comKaiser∗
2                    Google Research          University of Toronto              Google Brain
                      llion@google.com     aidan@cs.toronto.edu     

PAGE 3
Figure 1: The Transformer - model architecture.


The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the en

### Discussion: PDF Extraction Problems

When extracting text from scientific papers, at least five significant problems occur:

1. **Two-column layouts:** PyMuPDF's block sorting can interleave text from the left and right columns, producing unreadable output. For example, on a two-column page, a sentence from the left column may be followed by a sentence from the right column's header rather than the next paragraph.

2. **Mathematical equations:** Equations like `Attention(Q, K, V) = softmax(QK^T / √d_k)V` lose their spatial layout and become fragmented text. Subscripts, superscripts, and special operators (∑, ∏, √) are extracted as isolated Unicode characters without their mathematical context.

3. **Headers and footers:** Running headers (e.g., the conference name or paper title repeated on each page) and page numbers appear mixed into the body text. For instance, `"Proceedings of NIPS 2017"` may appear between two paragraphs of Section 3.

4. **Hyphenation at line breaks:** Words split across lines (e.g., `"atten-\ntion"`) are not automatically rejoined by the PDF extractor, resulting in tokens like `"atten-"` and `"tion"` being treated as separate words.

5. **Tables:** Table structure is lost during extraction. Cell contents are concatenated into lines of text, making it impossible to reconstruct the original rows and columns. A results table may appear as `"28.4 3.5 days 8 GPUs"` without column headers.

---
## Task 2 — Section Detection

The section detector combines multiple heuristics to identify headings across all papers without paper-specific rules:

- **Numbered heading regex** — matches patterns like `1`, `3.2`, `A.1`
- **Typography cues** — checks for bold fonts, larger-than-body font size
- **Small-caps fallback** — detects uppercase headings common in conference templates
- **Common unnumbered labels** — recognizes `Abstract`, `References`, `Acknowledgments`

The detector uses PyMuPDF's `dict` output to access font metadata, and sorts candidates by reading order (left column first on two-column pages).

In [5]:
from scipaper.sections import detect_sections
import json

# Detect sections for all papers
all_sections = {}
for pdf_path in PDF_FILES:
    sections = detect_sections(pdf_path)
    all_sections[pdf_path.stem] = sections
    print(f"{pdf_path.stem}: {len(sections)} section candidates detected")

print(f"\nTotal: {sum(len(s) for s in all_sections.values())} heading candidates across {len(PDF_FILES)} papers")

1406.1078v3: 23 section candidates detected


1409.0473v7: 33 section candidates detected


1412.3555v1: 14 section candidates detected


1601.06733v7: 14 section candidates detected


1607.06450v1: 22 section candidates detected


1610.02357v3: 19 section candidates detected
1703.03906v2: 20 section candidates detected


AttentionYouNeed: 25 section candidates detected


N16-1024: 22 section candidates detected

Total: 192 heading candidates across 9 papers


In [6]:
# Show sections detected in 'Attention Is All You Need' as an example
print("Sections detected in AttentionYouNeed.pdf:\n")
for sec in all_sections.get('AttentionYouNeed', []):
    indent = "    " if sec['section_id'] and '.' in str(sec['section_id']) else ""
    sid = sec['section_id'] or ""
    print(f"{indent}{sid} {sec['title']} (page {sec['page']})")

Sections detected in AttentionYouNeed.pdf:

 Abstract (page 1)
1 Introduction (page 2)
2 Background (page 2)
3 Model Architecture (page 2)
    3.1 Encoder and Decoder Stacks (page 3)
    3.2 Attention (page 3)
    3.2.1 Scaled Dot-Product Attention (page 4)
    3.2.2 Multi-Head Attention (page 4)
    3.2.3 Applications of Attention in our Model (page 5)
    3.3 Position-wise Feed-Forward Networks (page 5)
    3.4 Embeddings and Softmax (page 5)
    3.5 Positional Encoding (page 6)
4 Why Self-Attention (page 6)
5 Training (page 7)
    5.1 Training Data and Batching (page 7)
    5.2 Hardware and Schedule (page 7)
    5.3 Optimizer (page 7)
    5.4 Regularization (page 7)
6 Results (page 8)
    6.1 Machine Translation (page 8)
    6.2 Model Variations (page 8)
    6.3 English Constituency Parsing (page 9)
7 Conclusion (page 10)
 Acknowledgements (page 10)
 References (page 10)


---
## Task 3 — Paragraph Detection

The paragraph detector uses **PyMuPDF layout blocks** as candidates, then applies several filters:

**How paragraph boundaries are determined:**
1. Each PDF text block becomes a candidate paragraph
2. Recognized headings (from Task 2) are excluded and used to assign section labels
3. Captions (`Figure 1:`, `Table 2:`) are excluded using regex patterns
4. Page numbers, equations, and short non-prose fragments are filtered out
5. Front-matter blocks on page 1 (title, authors, affiliations) before the first heading are excluded
6. Footer/footnote blocks at the bottom of pages with smaller font size are excluded
7. Adjacent blocks on the same page, column, and section are **merged** when the previous block lacks sentence-ending punctuation and the next block starts with a lowercase letter
8. References and Acknowledgments sections are excluded from the body paragraphs

All excluded blocks and their reasons are retained in the output for auditing.

In [7]:
from scipaper.paragraphs import extract_document_paragraphs
import json

paragraph_dir = PROJECT_ROOT / "outputs" / "paragraphs"
paragraph_dir.mkdir(parents=True, exist_ok=True)

paragraph_summaries = []
for pdf_path in PDF_FILES:
    result = extract_document_paragraphs(pdf_path)
    output_path = paragraph_dir / f"{pdf_path.stem}_paragraphs.json"
    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    paragraph_summaries.append({
        "paper": result["paper"],
        **result["statistics"],
        "output": str(output_path)
    })
    excluded_types = result["statistics"]["excluded_by_type"]
    print(f"{pdf_path.stem}: {result['statistics']['paragraphs']} paragraphs, "
          f"{result['statistics']['excluded_candidates']} excluded")
    print(f"  Excluded types: {excluded_types}")

total_paragraphs = sum(s["paragraphs"] for s in paragraph_summaries)
print(f"\nTotal: {total_paragraphs} paragraphs across {len(PDF_FILES)} papers")

1406.1078v3: 152 paragraphs, 275 excluded
  Excluded types: {'caption': 10, 'equation': 32, 'excluded_section_acknowledgments': 2, 'excluded_section_references': 47, 'footer_or_footnote': 8, 'front_matter': 6, 'heading': 22, 'numeric_data_block': 2, 'repeated_header_footer_or_title': 2, 'short_fragment': 3, 'short_nonprose_fragment': 132, 'table': 9}


1409.0473v7: 119 paragraphs, 327 excluded
  Excluded types: {'caption': 6, 'equation': 41, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 42, 'footer_or_footnote': 5, 'front_matter': 4, 'heading': 33, 'numeric_data_block': 2, 'page_number': 15, 'repeated_header_footer_or_title': 15, 'short_fragment': 7, 'short_nonprose_fragment': 143, 'table': 13}


1412.3555v1: 86 paragraphs, 160 excluded
  Excluded types: {'caption': 5, 'equation': 23, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 34, 'footer_or_footnote': 3, 'front_matter': 2, 'front_matter_title_or_author_block': 3, 'heading': 14, 'numeric_data_block': 6, 'page_number': 9, 'short_fragment': 2, 'short_nonprose_fragment': 58}


1601.06733v7: 82 paragraphs, 162 excluded
  Excluded types: {'caption': 6, 'equation': 18, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 31, 'footer_or_footnote': 1, 'front_matter': 1, 'heading': 14, 'numeric_data_block': 2, 'short_fragment': 1, 'short_nonprose_fragment': 87}


1607.06450v1: 131 paragraphs, 468 excluded
  Excluded types: {'caption': 8, 'equation': 49, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 113, 'footer_or_footnote': 2, 'front_matter': 3, 'front_matter_title_or_author_block': 3, 'heading': 22, 'numeric_data_block': 10, 'page_number': 13, 'short_fragment': 2, 'short_nonprose_fragment': 235, 'table': 7}


1610.02357v3: 68 paragraphs, 53 excluded
  Excluded types: {'caption': 13, 'excluded_section_references': 11, 'footer_or_footnote': 3, 'front_matter': 4, 'heading': 17, 'numeric_data_block': 1, 'short_nonprose_fragment': 4}


1703.03906v2: 78 paragraphs, 111 excluded
  Excluded types: {'caption': 10, 'equation': 9, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 39, 'footer_or_footnote': 3, 'heading': 20, 'numeric_data_block': 7, 'page_number': 1, 'short_nonprose_fragment': 21}


AttentionYouNeed: 85 paragraphs, 454 excluded
  Excluded types: {'caption': 9, 'equation': 12, 'excluded_section_acknowledgements': 1, 'excluded_section_references': 79, 'footer_or_footnote': 2, 'front_matter': 15, 'heading': 24, 'numeric_data_block': 1, 'page_number': 14, 'short_fragment': 10, 'short_nonprose_fragment': 279, 'table': 8}


N16-1024: 127 paragraphs, 105 excluded
  Excluded types: {'caption': 9, 'equation': 11, 'excluded_section_acknowledgments': 1, 'excluded_section_references': 25, 'footer_or_footnote': 4, 'front_matter': 3, 'heading': 22, 'numeric_data_block': 2, 'page_number': 11, 'short_nonprose_fragment': 17}

Total: 928 paragraphs across 9 papers


In [8]:
# Show example paragraphs from the main paper
result = extract_document_paragraphs(PAPER_DIR / 'AttentionYouNeed.pdf')
print("First 5 paragraph records from Attention Is All You Need:\n")
for para in result['paragraphs'][:5]:
    section = f"{para.get('section_id', '')} {para.get('section', 'Unassigned')}".strip()
    print(f"  P{para['paragraph']} | page {para['page']} | {section}")
    print(f"    {para['text'][:120]}...")
    print()

First 5 paragraph records from Attention Is All You Need:

  P1 | page 1 | None Abstract
    The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include a...

  P2 | page 1 | None Abstract
    ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort t...

  P3 | page 1 | None Abstract
    31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA....

  P4 | page 2 | 1 Introduction
    Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been ...

  P5 | page 2 | 1 Introduction
    Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the...



---
## Task 4 — Sentence Segmentation

We use **spaCy's trained English parser** (`en_core_web_sm`) for sentence boundary detection, plus a narrow abbreviation repair rule.

### Why `text.split(".")` is inadequate for scientific papers

`text.split(".")` treats every period as a sentence boundary. This fails in at least five common cases in scientific text:

1. **Abbreviations:** `"et al."` would be split into `"et al"` + `""`, breaking citation patterns like `"Vaswani et al. proposed..."` into two fragments.

2. **Decimal numbers:** `"achieves 28.4 BLEU"` would be split at the period, producing `"achieves 28"` and `"4 BLEU"` — two meaningless fragments from one sentence.

3. **Citations and references:** `"See Sec. 3.2 for details"` contains three periods, none of which is a sentence boundary. `split(".")` would produce four fragments.

4. **Parenthetical expressions:** `"(Devlin et al., 2014)"` would split inside the parentheses, breaking the citation structure.

5. **Equations and formulas:** `"We compute Eq. (1) where d_k = 64."` would split after `"Eq"` rather than at the actual sentence end.

Our pipeline uses spaCy's parser, which considers syntactic structure, and adds a conservative repair for known abbreviations followed by parentheses (e.g., joining `"et al."` + `"(2017)"` when the parser incorrectly splits them).

In [9]:
import spacy
from scipaper.sentences import process_file

nlp = spacy.load("en_core_web_sm")

# Demonstrate why text.split(".") fails
demo_text = "Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task. See Sec. 3.2 for details."
print("Demo sentence:", demo_text)
print(f"\ntext.split('.') produces {len(demo_text.split('.'))} fragments:")
for i, frag in enumerate(demo_text.split('.')):
    print(f"  Fragment {i+1}: '{frag.strip()}'")

print(f"\nspaCy parser produces {len(list(nlp(demo_text).sents))} sentences:")
for i, sent in enumerate(nlp(demo_text).sents):
    print(f"  Sentence {i+1}: '{sent.text.strip()}'")

Demo sentence: Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task. See Sec. 3.2 for details.

text.split('.') produces 7 fragments:
  Fragment 1: 'Vaswani et al'
  Fragment 2: '(2017) achieve 28'
  Fragment 3: '4 BLEU on the WMT 2014 task'
  Fragment 4: 'See Sec'
  Fragment 5: '3'
  Fragment 6: '2 for details'
  Fragment 7: ''

spaCy parser produces 2 sentences:
  Sentence 1: 'Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task.'
  Sentence 2: 'See Sec. 3.2 for details.'


In [10]:
# Run Task 4 on all paragraph JSON files
paragraph_dir = PROJECT_ROOT / "outputs" / "paragraphs"
sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
paragraph_jsons = sorted(paragraph_dir.glob("*_paragraphs.json"))

sentence_results = [process_file(path, nlp, sentence_dir) for path in paragraph_jsons]

for r in sentence_results:
    print(f"{r['paper']}: {r['sentences']} sentences from {r['paragraphs']} paragraphs")

total_sentences = sum(r['sentences'] for r in sentence_results)
print(f"\nTotal: {total_sentences} sentences across {len(sentence_results)} papers")

1406.1078v3: 328 sentences from 152 paragraphs
1409.0473v7: 276 sentences from 119 paragraphs
1412.3555v1: 185 sentences from 86 paragraphs
1601.06733v7: 250 sentences from 82 paragraphs
1607.06450v1: 330 sentences from 131 paragraphs
1610.02357v3: 186 sentences from 68 paragraphs
1703.03906v2: 196 sentences from 78 paragraphs
AttentionYouNeed: 233 sentences from 85 paragraphs
N16-1024: 301 sentences from 127 paragraphs

Total: 2285 sentences across 9 papers


---
## Task 5 — Tokenization and Linguistic Annotations

Each corrected Task 4 sentence is parsed independently by spaCy. For every token we record:
- **text** — the surface form
- **token_index** — zero-based position within the sentence
- **lemma** — the dictionary base form
- **pos** — coarse part-of-speech tag
- **dependency** — syntactic dependency relation
- **head** — the syntactic head token

We also add a small tokenizer extension: two infix rules that split parentheses attached to letters (e.g., `p(xt)` → `p`, `(`, `xt`, `)`). This improves token boundaries around inline mathematical notation without affecting other English tokenization.

In [11]:
import spacy
from scipaper.tokens import configure_scientific_tokenizer, process_file as token_process_file

task5_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(task5_nlp)

sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
token_dir = PROJECT_ROOT / "outputs" / "tokens"
sentence_jsons = sorted(sentence_dir.glob("*_sentences.json"))

token_results = [token_process_file(path, task5_nlp, token_dir) for path in sentence_jsons]

for r in token_results:
    print(f"{r['paper']}: {r['tokens']} tokens from {r['sentences']} sentences")

total_tokens = sum(r['tokens'] for r in token_results)
print(f"\nTotal: {total_tokens} tokens across {len(token_results)} papers")

1406.1078v3: 6670 tokens from 328 sentences
1409.0473v7: 6659 tokens from 276 sentences
1412.3555v1: 3512 tokens from 185 sentences
1601.06733v7: 5746 tokens from 250 sentences
1607.06450v1: 5996 tokens from 330 sentences
1610.02357v3: 3962 tokens from 186 sentences
1703.03906v2: 4476 tokens from 196 sentences
AttentionYouNeed: 4895 tokens from 233 sentences
N16-1024: 6152 tokens from 301 sentences

Total: 48068 tokens across 9 papers


In [12]:
# Show token example from the first sentence of Attention Is All You Need
import json

token_path = token_dir / "AttentionYouNeed_tokens.json"
token_doc = json.loads(token_path.read_text(encoding="utf-8"))
first_sentence = token_doc["paragraphs"][0]["sentences"][0]

print(f"Sentence: {first_sentence['text']}\n")
print(f"{'Token':<20} {'Lemma':<20} {'POS':<8} {'Dependency':<12} {'Head'}")
print("-" * 80)
for tok in first_sentence["tokens"][:12]:
    print(f"{tok['text']:<20} {tok['lemma']:<20} {tok['pos']:<8} {tok['dependency']:<12} {tok['head']}")
if len(first_sentence["tokens"]) > 12:
    print(f"... ({len(first_sentence['tokens'])} tokens total)")

Sentence: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.

Token                Lemma                POS      Dependency   Head
--------------------------------------------------------------------------------
The                  the                  DET      det          models
dominant             dominant             ADJ      amod         models
sequence             sequence             NOUN     compound     transduction
transduction         transduction         NOUN     compound     models
models               model                NOUN     nsubjpass    based
are                  be                   AUX      auxpass      based
based                base                 VERB     ROOT         based
on                   on                   ADP      prep         based
complex              complex              ADJ      amod         recurrent
recurrent            recurrent            NOUN    

---
## Task 6 — Part-of-Speech Analysis

POS statistics use non-punctuation tokens as the denominator (punctuation is excluded from the nine required POS categories but its count is reported separately). We compare three papers to observe differences in their linguistic profiles.

In [13]:
import pandas as pd
from scipaper.pos_stats import process_files

token_dir = PROJECT_ROOT / "outputs" / "tokens"
statistics_dir = PROJECT_ROOT / "outputs" / "statistics"
token_jsons = sorted(token_dir.glob("*_tokens.json"))
pos_rows = process_files(token_jsons, statistics_dir)

REQUIRED_POS = ("NOUN", "VERB", "ADJ", "ADV", "PROPN", "PRON", "DET", "ADP", "NUM")

# Show required POS percentages for every paper.
all_paper_rows = []
for row in pos_rows:
    entry = {"Paper": row["paper"], "Non-punctuation tokens": row["pos_percentage_denominator"]}
    for pos in REQUIRED_POS:
        entry[pos] = f"{row['pos_percentages'][pos]:.1f}%"
    all_paper_rows.append(entry)

print(f"Required POS percentages — all {len(all_paper_rows)} papers")
display(pd.DataFrame(all_paper_rows))

# Compare counts and percentages for three papers, as required by the assignment.
selected_papers = {"AttentionYouNeed", "N16-1024", "1607.06450v1"}
comparison_rows = []
for row in pos_rows:
    if row["paper"] in selected_papers:
        entry = {"Paper": row["paper"]}
        for pos in REQUIRED_POS:
            entry[f"{pos} Count"] = row["pos_counts"][pos]
            entry[f"{pos} %"] = f"{row['pos_percentages'][pos]:.1f}%"
        entry["Total (excl. punct)"] = row["pos_percentage_denominator"]
        comparison_rows.append(entry)

print("Detailed POS count and percentage comparison — three papers")
for row in comparison_rows:
    print(f"\n=== {row['Paper']} (total non-punct tokens: {row['Total (excl. punct)']:,}) ===")
    print(f"{'POS':<8} {'Count':>8} {'Percentage':>12}")
    print("-" * 30)
    for pos in REQUIRED_POS:
        print(f"{pos:<8} {row[f'{pos} Count']:>8,} {row[f'{pos} %']:>12}")


Required POS percentages — all 9 papers


,Paper,Non-punctuation tokens,NOUN,VERB,ADJ,ADV,PROPN,PRON,DET,ADP,NUM
0,1406.1078v3,5639,24.0%,10.9%,7.3%,3.4%,12.0%,3.4%,12.9%,10.8%,3.0%
1,1409.0473v7,5598,24.8%,11.4%,7.4%,3.7%,9.9%,3.6%,14.2%,11.2%,2.8%
2,1412.3555v1,2971,23.9%,9.9%,9.8%,4.5%,9.0%,3.7%,12.8%,11.3%,3.2%
3,1601.06733v7,4871,26.5%,10.7%,9.5%,3.0%,10.4%,4.1%,9.7%,11.1%,4.5%
4,1607.06450v1,5282,27.9%,11.1%,7.7%,2.9%,8.3%,3.2%,11.5%,11.2%,4.1%
5,1610.02357v3,3508,25.4%,9.4%,11.0%,3.9%,7.3%,4.1%,9.4%,13.1%,4.6%
6,1703.03906v2,3869,25.8%,11.5%,9.5%,4.3%,9.6%,5.1%,8.2%,11.0%,3.6%
7,AttentionYouNeed,4213,29.1%,10.5%,9.1%,3.4%,5.1%,4.1%,10.4%,12.6%,5.5%
8,N16-1024,5032,27.4%,10.2%,8.8%,3.3%,11.8%,3.9%,9.9%,10.4%,3.5%


Detailed POS count and percentage comparison — three papers

=== 1607.06450v1 (total non-punct tokens: 5,282) ===
POS         Count   Percentage
------------------------------
NOUN        1,475        27.9%
VERB          588        11.1%
ADJ           409         7.7%
ADV           155         2.9%
PROPN         440         8.3%
PRON          171         3.2%
DET           606        11.5%
ADP           589        11.2%
NUM           214         4.1%

=== AttentionYouNeed (total non-punct tokens: 4,213) ===
POS         Count   Percentage
------------------------------
NOUN        1,225        29.1%
VERB          441        10.5%
ADJ           385         9.1%
ADV           145         3.4%
PROPN         214         5.1%
PRON          172         4.1%
DET           438        10.4%
ADP           531        12.6%
NUM           232         5.5%

=== N16-1024 (total non-punct tokens: 5,032) ===
POS         Count   Percentage
------------------------------
NOUN        1,381        27.4%
VER

### Three-Paper POS Comparison

The three papers show broadly similar POS distributions — as expected for scientific English — but with notable differences:

- **NOUN** is the most frequent category in all three papers (~27–29%), reflecting the noun-heavy nature of scientific writing
- **PROPN (proper nouns)** varies significantly: `N16-1024` has the highest share because it discusses many named systems, models, and benchmarks
- **NUM** is highest in `AttentionYouNeed` because of its extensive results tables with BLEU scores and training parameters
- **VERB** is relatively low (~10–11%) compared to general English, as scientific papers favor nominal constructions

These differences reflect the content and style of each paper rather than fundamental linguistic properties.

---
## Task 7 — Dependency Parsing and Visualization

We selected 10 sentences from 10 distinct sections across all nine papers. For each sentence, we display token, lemma, POS, dependency label, and syntactic head. Three sentences are visualized with displaCy and their structures are briefly explained.

In [14]:
import json
import pandas as pd
import spacy
from IPython.display import HTML, display
from scipaper.dependencies import build_dependency_examples
from scipaper.tokens import configure_scientific_tokenizer

dependency_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(dependency_nlp)

dependency_manifest = PROJECT_ROOT / "data" / "annotations" / "task7_dependency_examples.json"
dependency_output_dir = PROJECT_ROOT / "outputs" / "dependency_examples"

dependency_summary = build_dependency_examples(
    dependency_manifest,
    PROJECT_ROOT / "outputs" / "tokens",
    dependency_output_dir,
    dependency_nlp,
)
print(f"Generated {dependency_summary['samples']} dependency examples")
print(f"  Distinct sections: {dependency_summary['distinct_sections']}")
print(f"  Token rows: {dependency_summary['token_rows']}")
print(f"  Visualizations: {len(dependency_summary['visualizations'])}")

Generated 10 dependency examples
  Distinct sections: 10
  Token rows: 238
  Visualizations: 3


In [15]:
# Display the dependency token table
dependency_csv = dependency_output_dir / "dependency_tokens.csv"
dep_table = pd.read_csv(dependency_csv)
display(dep_table[["case_id", "section", "token", "lemma", "pos", "dependency", "head"]])

,case_id,section,token,lemma,pos,dependency,head
0,passive_training_objective,Abstract,The,the,DET,det,encoder
1,passive_training_objective,Abstract,encoder,encoder,NOUN,nsubjpass,trained
2,passive_training_objective,Abstract,and,and,CCONJ,cc,encoder
3,passive_training_objective,Abstract,decoder,decoder,NOUN,conj,encoder
4,passive_training_objective,Abstract,of,of,ADP,prep,encoder
...,...,...,...,...,...,...,...
233,transition_sequence,Transition Sequences from Trees,of,of,ADP,prep,traversal
234,transition_sequence,Transition Sequences from Trees,a,a,DET,det,tree
235,transition_sequence,Transition Sequences from Trees,parse,parse,NOUN,compound,tree
236,transition_sequence,Transition Sequences from Trees,tree,tree,NOUN,pobj,of


### Three Dependency Structure Explanations

**1. Passive training objective** (`1406.1078v3`, Abstract):
The sentence uses a passive construction where `"trained"` is the ROOT verb. The encoder and decoder are coordinated passive subjects (`nsubjpass`), joined by `"and"` (cc/conj). The infinitive `"maximize"` functions as an adverbial complement of purpose (xcomp), and `"probability"` is its direct object (dobj). This tree shows how spaCy correctly identifies the passive voice and purpose clause.

**2. Relative clause** (`1601.06733v7`, The Machine Reader):
`"present"` is the ROOT, with `"we"` as its subject (nsubj) and `"reader"` as its direct object (dobj). The relative clause `"which is designed to..."` modifies `"reader"` — `"which"` is the subject (nsubjpass) of the passive `"designed"`, and `"process"` is an open clausal complement (xcomp). The diagram illustrates how spaCy handles relative clause attachment.

**3. Coordinated measures** (`1703.03906v2`, Introduction):
`"report"` is the ROOT with `"We"` as subject. The four reported measures form a coordinated list joined by commas and `"and"`. spaCy may attach list items imperfectly — this is a known limitation with long coordinated structures. The participial phrase `"including variance numbers"` attaches as an adverbial modifier.

In [16]:
# Display displaCy visualizations
from spacy import displacy
from IPython.display import HTML

manifest = json.loads(dependency_manifest.read_text(encoding="utf-8"))
visualize_cases = [c for c in manifest["cases"] if c.get("visualize")]

for case in visualize_cases:
    # Read the sentence text from the dependency examples output
    dep_examples = json.loads((dependency_output_dir / "dependency_examples.json").read_text(encoding="utf-8"))
    example = next(e for e in dep_examples["examples"] if e["case_id"] == case["case_id"])
    
    print(f"\n{'='*80}")
    print(f"Case: {case['case_id']} | Paper: {case['paper']} | Section: {case['section']}")
    print(f"Sentence: {example['text']}")
    print(f"{'='*80}")
    
    doc = dependency_nlp(example["text"])
    html = displacy.render(doc, style="dep", jupyter=False, options={"compact": True, "distance": 100})
    display(HTML(html))


Case: passive_training_objective | Paper: 1406.1078v3 | Section: Abstract
Sentence: The encoder and decoder of the proposed model are jointly trained to maximize the conditional probability of a target sequence given a source sequence.



Case: machine_reader_relative_clause | Paper: 1601.06733v7 | Section: The Machine Reader
Sentence: In this section we present our machine reader which is designed to process structured input while retaining the incrementality of a recurrent neural network.



Case: reporting_experimental_results | Paper: 1703.03906v2 | Section: Introduction
Sentence: We report BLEU scores, perplexities, model sizes, and convergence time for all experiments, including variance numbers calculated across several runs of each experiment.


---
## Task 8 — Document Statistics

The table below summarizes key metrics for each paper. Definitions:
- **Tokens** includes punctuation (total spaCy tokens)
- **Unique Tokens** are case-folded surface forms excluding punctuation
- **Avg Sentence Length** and **Avg Paragraph Length** count non-punctuation tokens

In [17]:
import pandas as pd
from scipaper.document_stats import process_files

token_files = sorted((PROJECT_ROOT / "outputs" / "tokens").glob("*_tokens.json"))
statistics_dir = PROJECT_ROOT / "outputs" / "statistics"
document_stats_rows = process_files(token_files, statistics_dir)

df = pd.DataFrame(document_stats_rows)
df.columns = [c.replace("_", " ").title() for c in df.columns]

# Keep the required cross-paper comparison narrow enough to read in VS Code.
required_columns = [
    "Paper", "Pages", "Sections Detected", "Paragraphs", "Sentences",
    "Tokens Including Punctuation",
]
print(f"Required document statistics — all {len(df)} papers")
display(df[required_columns])

# The assignment also asks for lexical diversity and average lengths.
extra_columns = {
    "Paper": "Paper",
    "Unique Tokens Casefolded Excluding Punctuation": "Unique lexical forms",
    "Average Sentence Length Lexical Tokens": "Avg. lexical tokens / sentence",
    "Average Paragraph Length Lexical Tokens": "Avg. lexical tokens / paragraph",
}
print("Additional vocabulary and length statistics — all papers")
display(df[list(extra_columns)].rename(columns=extra_columns))

print(f"\n{'='*60}")
print(f"TOTALS across {len(df)} papers:")
print(f"  Pages:      {df['Pages'].sum()}")
print(f"  Paragraphs: {df['Paragraphs'].sum()}")
print(f"  Sentences:  {df['Sentences'].sum()}")
print(f"  Tokens:     {df['Tokens Including Punctuation'].sum():,}")


Required document statistics — all 9 papers


,Paper,Pages,Sections Detected,Paragraphs,Sentences,Tokens Including Punctuation
0,1406.1078v3,15,23,152,328,6670
1,1409.0473v7,15,33,119,276,6659
2,1412.3555v1,9,14,86,185,3512
3,1601.06733v7,11,14,82,250,5746
4,1607.06450v1,14,22,131,330,5996
5,1610.02357v3,8,19,68,186,3962
6,1703.03906v2,9,20,78,196,4476
7,AttentionYouNeed,15,25,85,233,4895
8,N16-1024,11,22,127,301,6152


Additional vocabulary and length statistics — all papers


,Paper,Unique lexical forms,Avg. lexical tokens / sentence,Avg. lexical tokens / paragraph
0,1406.1078v3,1132,17.19,37.10
1,1409.0473v7,1135,20.28,47.04
2,1412.3555v1,708,16.06,34.55
3,1601.06733v7,1237,19.48,59.40
4,1607.06450v1,1142,16.01,40.32
5,1610.02357v3,839,18.86,51.59
6,1703.03906v2,1030,19.74,49.60
7,AttentionYouNeed,1075,18.08,49.56
8,N16-1024,1120,16.72,39.62



TOTALS across 9 papers:
  Pages:      107
  Paragraphs: 928
  Sentences:  2285
  Tokens:     48,068


---
## Task 9 — Final Nested JSON Representation

The pipeline produces one structured JSON file per paper with the hierarchy:

```
Paper
  → Section (section_id, title)
    → Paragraph (paragraph_id, page, text)
      → Sentence (sentence_id, text)
        → Token (text, token_index, lemma, pos, dependency, head)
```

This output will be used as input for Homework 2.

In [18]:
from scipaper.final_json import process_files as build_final_json

token_files = sorted((PROJECT_ROOT / "outputs" / "tokens").glob("*_tokens.json"))
final_summaries = build_final_json(token_files, PROJECT_ROOT / "outputs" / "json")

print("Generated JSON files:\n")
for s in final_summaries:
    print(f"  {s['paper']}: {s['sections_with_paragraphs']} sections, "
          f"{s['paragraphs']} paragraphs, {s['sentences']} sentences, "
          f"{s['tokens']} tokens")
    print(f"    → {s['output']}")

Generated JSON files:

  1406.1078v3: 20 sections, 152 paragraphs, 328 sentences, 6670 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1406.1078v3.json
  1409.0473v7: 23 sections, 119 paragraphs, 276 sentences, 6659 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1409.0473v7.json
  1412.3555v1: 10 sections, 86 paragraphs, 185 sentences, 3512 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1412.3555v1.json
  1601.06733v7: 12 sections, 82 paragraphs, 250 sentences, 5746 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1601.06733v7.json
  1607.06450v1: 19 sections, 131 paragraphs, 330 sentences, 5996 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1607.06450v1.json
  1610.02357v3: 15 sections, 68 paragraphs, 186 sentences, 3962 tokens
    → /Users/hilalhocaoglu/Documents/4.sınıf_guz/nlp 2/outputs/json/1610.02357v3.json
  1703.03906v2: 16 sections, 78 paragr

In [19]:
# Preview the structure of the main paper's JSON
import json

preview_path = PROJECT_ROOT / "outputs" / "json" / "AttentionYouNeed.json"
attention_json = json.loads(preview_path.read_text(encoding="utf-8"))

print(f"Paper: {attention_json['paper']}")
print(f"Schema version: {attention_json['schema_version']}")
print(f"Sections: {len(attention_json['sections'])}")
print(f"\nSection titles:")
for sec in attention_json['sections']:
    para_count = len(sec['paragraphs'])
    sent_count = sum(len(p['sentences']) for p in sec['paragraphs'])
    print(f"  {sec['title']}: {para_count} paragraphs, {sent_count} sentences")

print(f"\n--- First sentence of first section (preview) ---")
first_sent = attention_json['sections'][0]['paragraphs'][0]['sentences'][0]
print(f"Text: {first_sent['text'][:100]}...")
print(f"Tokens: {len(first_sent['tokens'])} tokens")
print(f"First 5 tokens: {[t['text'] for t in first_sent['tokens'][:5]]}")

Paper: AttentionYouNeed
Schema version: 1.0
Sections: 22

Section titles:
  Abstract: 3 paragraphs, 16 sentences
  1 Introduction: 4 paragraphs, 13 sentences
  2 Background: 4 paragraphs, 12 sentences
  3 Model Architecture: 2 paragraphs, 6 sentences
  3.1 Encoder and Decoder Stacks: 2 paragraphs, 12 sentences
  3.2 Attention: 4 paragraphs, 6 sentences
  3.2.1 Scaled Dot-Product Attention: 4 paragraphs, 14 sentences
  3.2.2 Multi-Head Attention: 8 paragraphs, 15 sentences
  3.2.3 Applications of Attention in our Model: 1 paragraphs, 1 sentences
  3.3 Position-wise Feed-Forward Networks: 5 paragraphs, 16 sentences
  3.4 Embeddings and Softmax: 1 paragraphs, 4 sentences
  3.5 Positional Encoding: 4 paragraphs, 11 sentences
  4 Why Self-Attention: 7 paragraphs, 24 sentences
  5 Training: 1 paragraphs, 1 sentences
  5.1 Training Data and Batching: 1 paragraphs, 5 sentences
  5.2 Hardware and Schedule: 1 paragraphs, 5 sentences
  5.3 Optimizer: 3 paragraphs, 6 sentences
  5.4 Regularization

---
## Summary

This notebook processed **9 scientific PDFs** through a complete NLP pipeline:

| Stage | Output |
|---|---|
| Task 1 — PDF Extraction | Page-marked text files |
| Task 2 — Section Detection | Heading candidates with page numbers |
| Task 3 — Paragraph Detection | Filtered, section-labeled paragraph records |
| Task 4 — Sentence Segmentation | spaCy sentences with abbreviation repair |
| Task 5 — Tokenization | Tokens with lemma, POS, dependency, head |
| Task 6 — POS Analysis | Per-paper POS counts and percentages |
| Task 7 — Dependency Parsing | 10 examples, 3 displaCy visualizations |
| Task 8 — Document Statistics | Cross-paper comparison table |
| Task 9 — JSON Output | One nested JSON file per paper |

All outputs are in `outputs/`. The experiment log (`notes/experiment_log.md`) documents the methods tried, what worked, and what didn't at each stage.